In [1]:
from Data_query.trino_config import *
import json
import numpy as np
import boto3
from sklearn.neighbors import KDTree
session = boto3.Session()
s3 = boto3.client('s3')
import random

In [10]:
stop_trino()

Trino service stopping triggered.


In [2]:
big_workers = 2
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count = workers, big_worker_desired_count=big_workers)
sleep(90)


Trino service is already running.


In [6]:
def Q_impact(row):
    r1 = abs(row['Q_kvar']) / (abs(row['Q_voltvar_max_final']) + 1e-9)
    r2 = abs(row['Q_kvar']) / (abs(row['Q_voltvar_min_final']) + 1e-9)

    # choose the smaller one, but keep track of which was chosen
    if r1 <= r2:
        chosen = r1
        sign = np.sign(row['Q_voltvar_max_final']) * np.sign(row['Q_kvar'])
    else:
        chosen = r2
        sign = np.sign(row['Q_voltvar_min_final']) * np.sign(row['Q_kvar'])

    if row['Q_voltvar_max_final'] + row['Q_voltvar_min_final'] == 0:
        sign = 1
    return chosen * sign

In [7]:
def get_voltvar_Q(V, Srated=1, v1=207, v2=220, v3=240, v4=258, Q1=.44, Q4=.60):
    if V <= v1:
        Q = Q1* Srated
    elif v1 <= V < v2:
        m = (Q1* Srated - 0) / (v1 - v2)
        Q = m * (V - v2)
    elif v2 <= V <= v3:
        Q = float(0)
    elif v3 < V < v4:
        m = (0 - Q4* Srated) / (v3 - v4)
        Q = -m * (V - v4) - Q4* Srated
    else:  # V >= v4
        Q = - Q4* Srated

    return Q

def Q_capability_absorbing(P, Srated=1):
    if abs(P) < .2 * Srated:
        Q = float(0)
    elif abs(P) <= .6 * Srated:
        Q = - 0.44 * Srated
    elif abs(P) <= .8 * Srated:
        S_pf = abs(P) / 0.8
        val = S_pf**2 - abs(P)**2
        if val < 0:   # protect against negatives
            Q = float(0)
        else:
            Q = -math.sqrt(val)
    else:
        val = Srated**2 - abs(P)**2
        if val < 0:   # protect against negatives
            Q = float(0)
        else:
            Q = -math.sqrt(val)
    return Q

In [3]:
iceberg_exec("DROP TABLE IF EXISTS conformance_voltvar")
iceberg_exec("""CREATE TABLE conformance_voltvar (
                year BIGINT,
                month BIGINT,
                day BIGINT,
                day_night VARCHAR,
                site_id BIGINT,
                P_kW_sum DOUBLE,
                nonconformance_voltvar_sum DOUBLE,
                Q_adverse_sum DOUBLE,
                Q_inactive_sum DOUBLE,
                Q_minor_deviation_sum DOUBLE,
                Q_major_deficit_sum DOUBLE,
                Q_major_surplus_sum DOUBLE,
                curtailment_voltvar_sum DOUBLE,
                nonconformance_voltvar_count BIGINT,
                Q_adverse_count BIGINT,
                Q_inactive_count BIGINT,
                Q_minor_deviation_count BIGINT,
                Q_major_deficit_count BIGINT,
                Q_major_surplus_count BIGINT,
                curtailment_voltvar_count BIGINT,
                total_count BIGINT
            )
    """)

Executed
Executed


In [4]:
sleep(10)

In [5]:
v1=207
v2=220
v3=240
v4=258
voltwatt_V = 253
Q1=.44
Q4=.60
thr1 = -0.1
thr2 = 0.1
thr3 = .9
thr4 = 1.1
def run_func(args):
    year, month, split_cons = args
    iceberg_exec(f"""
                insert into conformance_voltvar
                with 
                data as (
                    select site_id, t_stamp, sum(power*circuit_polarity/1000) as P_kW, 
                        sum(energy_reactive*circuit_polarity/1000*12) as Q_kvar, 
                            avg(voltage) as V, max(ac_capacity_kw) as ac_capacity_kw, max(S_99) as S_99
                    from 
                        (select circuit_id, t_stamp, power, energy_reactive, voltage 
                            from ts where year={year} and month={month} and is_pv=True and voltage > 0 and voltage < 300 and {split_cons} ) as ts1
                    inner join (select distinct site_id, circuit_id, circuit_polarity, ac_capacity_kw, S_99 from meta_up23c where is_pv = True) as m on ts1.circuit_id = m.circuit_id
                    group by site_id, t_stamp),
                pq as (
                    select site_id, t_stamp, P_kW, Q_kvar, V, ac_capacity_kw, S_99, 
                    case when V < {v1} then {Q1}*ac_capacity_kw 
                    when V < {v2} then ({Q1}* ac_capacity_kw - 0) / ({v1} - {v2}) * (V - {v2})
                    when V <= {v3} then 0
                    when V < {v4} then -(0 - {Q4}* ac_capacity_kw) / ({v3} - {v4}) * (V - {v4}) - {Q4}* ac_capacity_kw
                    else - {Q4}* ac_capacity_kw end as Q_voltvar,
                    case when abs(P_kW) < .2 * ac_capacity_kw then 0
                    when abs(P_kW) <= .6 * ac_capacity_kw then - 0.44 * ac_capacity_kw
                    when abs(P_kW) <= .8 * ac_capacity_kw then 
                        case when  (power(abs(P_kW)/0.8, 2) - power(abs(P_kW), 2) ) < 0 then 0
                        else -sqrt(power(abs(P_kW)/0.8, 2) - power(abs(P_kW), 2)) end
                    else 
                        case when ( power(ac_capacity_kw, 2) - power(abs(P_kW), 2) ) < 0 then 0
                        else -sqrt( power(ac_capacity_kw, 2) - power(abs(P_kW), 2) ) end
                    end as Q_cap_absorbing
                    from data
                    ),
                pq2 as (
                    select site_id, t_stamp, P_kW, Q_kvar, V, ac_capacity_kw, S_99, Q_voltvar, Q_cap_absorbing,
                    -Q_cap_absorbing as Q_cap_supplying,
                    Q_voltvar + .04 * ac_capacity_kw as Q_voltvar_max,
                        Q_voltvar - .04 * ac_capacity_kw as Q_voltvar_min
                    from pq
                    ),
                pq3 as (
                    select site_id, t_stamp, P_kW, Q_kvar, V, ac_capacity_kw, S_99, Q_voltvar, Q_cap_absorbing, Q_cap_supplying, Q_voltvar_max, Q_voltvar_min,
                    case when Q_voltvar_max < 0 then greatest(Q_voltvar_max, Q_cap_absorbing + .04*ac_capacity_kw) else Q_voltvar_max end as Q_voltvar_max_final,
                    case when Q_voltvar_min > 0 then least(Q_voltvar_min, Q_cap_supplying - .04*ac_capacity_kw) else Q_voltvar_min end as Q_voltvar_min_final
                    from pq2
                ),
                pq4 as (
                            select site_id, t_stamp, P_kW, Q_kvar, V, ac_capacity_kw, S_99, Q_voltvar_max_final, Q_voltvar_min_final, 
                            CASE 
                                WHEN abs(Q_kvar) / (abs(Q_voltvar_max_final) + 1e-9) 
                                    <= abs(Q_kvar) / (abs(Q_voltvar_min_final) + 1e-9)
                                THEN 
                                    CASE 
                                        WHEN Q_voltvar_max_final + Q_voltvar_min_final = 0 THEN 1
                                        ELSE sign(Q_voltvar_max_final) * sign(Q_kvar)
                                    END 
                                    * (abs(Q_kvar) / (abs(Q_voltvar_max_final) + 1e-9))
                                ELSE 
                                    CASE 
                                        WHEN Q_voltvar_max_final + Q_voltvar_min_final = 0 THEN 1
                                        ELSE sign(Q_voltvar_min_final) * sign(Q_kvar)
                                    END 
                                    * (abs(Q_kvar) / (abs(Q_voltvar_min_final) + 1e-9))
                                END AS Q_impact
                        from pq3),
                    pq5 as (
                        select pq4.site_id, pq4.t_stamp, P_kW, Q_kvar, V, ac_capacity_kw, S_99, Q_voltvar_max_final, Q_voltvar_min_final, Q_impact,
                        case when (Q_kvar < Q_voltvar_min_final or Q_kvar > Q_voltvar_max_final) and Q_impact < {thr1} then 
                        least(abs(Q_kvar - Q_voltvar_min_final), abs(Q_kvar - Q_voltvar_max_final)) else 0  end as Q_adverse,
                        case when (Q_kvar < Q_voltvar_min_final or Q_kvar > Q_voltvar_max_final) and (Q_impact >= {thr1} and Q_impact <= {thr2}) then 
                        least(abs(Q_kvar - Q_voltvar_min_final), abs(Q_kvar - Q_voltvar_max_final)) else 0 end as Q_inactive,
                        case when (Q_kvar < Q_voltvar_min_final or Q_kvar > Q_voltvar_max_final) and (Q_impact > {thr2} and Q_impact < {thr3}) then 
                        least(abs(Q_kvar - Q_voltvar_min_final), abs(Q_kvar - Q_voltvar_max_final)) else 0  end as Q_minor_deviation,
                        case when (Q_kvar < Q_voltvar_min_final or Q_kvar > Q_voltvar_max_final) and (Q_impact >= {thr3} and Q_impact <= {thr4}) then 
                        least(abs(Q_kvar - Q_voltvar_min_final), abs(Q_kvar - Q_voltvar_max_final)) else 0  end as Q_major_deficit,
                        case when (Q_kvar < Q_voltvar_min_final or Q_kvar > Q_voltvar_max_final) and (Q_impact > {thr4}) then 
                        least(abs(Q_kvar - Q_voltvar_min_final), abs(Q_kvar - Q_voltvar_max_final)) else 0  end as Q_major_surplus,
                        case when V <= {voltwatt_V} and sqrt(power(Q_kvar, 2) + power(P_kW, 2)) >= S_99 then uncurtailed_P - P_kW end as curtailment_voltvar
                        from pq4 left join (select site_id, t_stamp, uncurtailed_P from all_uncurtailedPV) as a on pq4.site_id = a.site_id and pq4.t_stamp = a.t_stamp
                    ),
                    pq6 as (
                        select site_id, t_stamp, P_kW, day(t_stamp) as day, month(t_stamp) as month, year(t_stamp) as year,
                        case when (hour(t_stamp) >= 20 or hour(t_stamp) <= 7) then 'day' else 'night' end as day_night,
                        Q_adverse, Q_inactive, Q_minor_deviation, Q_major_deficit, Q_major_surplus, 
                        Q_adverse + Q_inactive + Q_minor_deviation + Q_major_deficit + Q_major_surplus as nonconformance_voltvar,
                        curtailment_voltvar
                        from pq5
                    )
                select year, month, day, day_night, site_id, 
                sum(p_kW) as P_kW_sum, 
                sum(nonconformance_voltvar) as nonconformance_voltvar_sum,
                sum(Q_adverse) as Q_adverse_sum,
                sum(Q_inactive) as Q_inactive_sum,
                sum(Q_minor_deviation) as Q_minor_deviation_sum,
                sum(Q_major_deficit) as Q_major_deficit_sum,
                sum(Q_major_surplus) as Q_major_surplus_sum,
                sum(curtailment_voltvar) as curtailment_voltvar_sum,
                sum(case when nonconformance_voltvar > 0 then 1 else 0 end) as nonconformance_voltvar_count,
                sum(case when Q_adverse > 0 then 1 else 0 end) as Q_adverse_count,
                sum(case when Q_inactive > 0 then 1 else 0 end) as Q_inactive_count,
                sum(case when Q_minor_deviation > 0 then 1 else 0 end) as Q_minor_deviation_count,
                sum(case when Q_major_deficit > 0 then 1 else 0 end) as Q_major_deficit_count,
                sum(case when Q_major_surplus > 0 then 1 else 0 end) as Q_major_surplus_count,
                sum(case when curtailment_voltvar > 0 then 1 else 0 end) as curtailment_voltvar_count,
                count(nonconformance_voltvar) as total_count
                from pq6
                group by year, month, day, day_night, site_id
                """)
    sleep(random.randint(5, 15))
    print(f"Completed year={year}, month={month},  {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}")
    return None

tasks = [(year, month, split_cons) for year in (2024, 2025) for month in range(1, 13) 
         for split_cons in ['system.bucket(postcode, 16) <= 3', '(system.bucket(postcode, 16) > 3 and system.bucket(postcode, 16) <= 7)', 
                            '(system.bucket(postcode, 16) > 7 and system.bucket(postcode, 16) <= 11)', 'system.bucket(postcode, 16) > 11'] ]
trino_parallel_batch(run_func, tasks, num_workers=num_workers, batch_size=num_workers)

Executed
Executed
Completed year=2024, month=1,  bucket <= 3
Completed year=2024, month=1,  (bucket > 3 and bucket <= 7)
Executed
Executed
Completed year=2024, month=1,  (bucket > 7 and bucket <= 11)
Completed year=2024, month=1,  bucket > 11
Executed
Executed
Completed year=2024, month=2,  bucket <= 3
Completed year=2024, month=2,  (bucket > 3 and bucket <= 7)
Executed
Executed
Completed year=2024, month=2,  (bucket > 7 and bucket <= 11)
Completed year=2024, month=2,  bucket > 11
Executed
Executed
Completed year=2024, month=3,  (bucket > 3 and bucket <= 7)
Completed year=2024, month=3,  bucket <= 3
Executed
Executed
Completed year=2024, month=3,  (bucket > 7 and bucket <= 11)
Completed year=2024, month=3,  bucket > 11
Executed
Executed
Completed year=2024, month=4,  bucket <= 3
Completed year=2024, month=4,  (bucket > 3 and bucket <= 7)
Executed
Executed
Completed year=2024, month=4,  bucket > 11
Completed year=2024, month=4,  (bucket > 7 and bucket <= 11)
Executed
Executed
Completed 

In [6]:
iceberg_exec("""
    ALTER TABLE conformance_voltvar
    ADD COLUMN  nonconformance_voltvar_red_sum DOUBLE 
    """)
iceberg_exec("""
    ALTER TABLE conformance_voltvar
    ADD COLUMN  nonconformance_voltvar_red_count BIGINT
    """)

Executed
Executed


In [7]:
iceberg_exec(f"""
    UPDATE conformance_voltvar
        SET nonconformance_voltvar_red_sum = Q_adverse_sum + Q_inactive_sum + Q_major_deficit_sum,
            nonconformance_voltvar_red_count = Q_adverse_count + Q_inactive_count + Q_major_deficit_count
    """)

Executed


In [8]:
iceberg_exec("""
    ALTER TABLE conformance_voltvar
    ADD COLUMN  curtailment_voltvar_red_sum DOUBLE 
    """)
iceberg_exec("""
    ALTER TABLE conformance_voltvar
    ADD COLUMN  curtailment_voltvar_red_count BIGINT
    """)

Executed
Executed


In [9]:
iceberg_sql(f"""select  year, month, count( site_id)/1000 from conformance_voltvar
            group by year, month
            order by year, month
            """).round()


,year,month,_col2
0,2024,1,683
1,2024,2,656
2,2024,3,709
3,2024,4,693
4,2024,5,497
5,2024,6,686
6,2024,7,682
7,2024,8,679
8,2024,9,657
9,2024,10,680


In [18]:
iceberg_sql('select * from conformance_voltvar limit 10')

,year,month,day,day_night,site_id,nonconformance_voltvar_sum,q_adverse_sum,q_inactive_sum,q_minor_deviation_sum,q_major_deficit_sum,...,nonconformance_voltvar_count,q_adverse_count,q_inactive_count,q_minor_deviation_count,q_major_deficit_count,q_major_surplus_count,curtailment_voltvar_count,total_count,nonconformance_voltvar_red_sum,nonconformance_voltvar_red_count
0,2025,10,1,night,738575219,0.146654,0.00000,0.000000,0.0,0.008713,...,4,0,0,0,2,2,0,144,0.008713,2
1,2025,10,7,day,738575219,16.908075,0.00000,0.000000,0.0,0.017947,...,144,0,0,0,1,143,0,144,0.017947,1
2,2025,10,2,day,1026333277,208.651787,0.00000,208.651787,0.0,0.000000,...,110,0,110,0,0,0,0,144,208.651787,110
3,2025,10,3,night,1026333277,0.000000,0.00000,0.000000,0.0,0.000000,...,0,0,0,0,0,0,0,144,0.000000,0
4,2025,10,19,day,1026333277,377.251814,0.00000,377.251814,0.0,0.000000,...,124,0,124,0,0,0,0,144,377.251814,124
5,2025,10,31,night,1026333277,0.000000,0.00000,0.000000,0.0,0.000000,...,0,0,0,0,0,0,0,143,0.000000,0
6,2025,10,12,day,1033556736,15.523521,0.02153,15.501991,0.0,0.000000,...,50,1,49,0,0,0,0,144,15.523521,50
7,2025,10,16,night,1033556736,0.006143,0.00000,0.006143,0.0,0.000000,...,3,0,3,0,0,0,0,144,0.006143,3
8,2025,10,20,night,391185216,0.017316,0.00000,0.000000,0.0,0.017316,...,2,0,0,0,2,0,0,144,0.017316,2
9,2025,10,30,day,391185216,0.000000,0.00000,0.000000,0.0,0.000000,...,0,0,0,0,0,0,0,144,0.000000,0


In [14]:
iceberg_sql(f"""select  year, month, count( site_id)/1000 from conformance_voltvar
            group by year, month
            order by year, month
            """).round()


,year,month,_col2
0,2024,1,708
1,2024,2,681
2,2024,3,736
3,2024,4,720
4,2024,5,744
5,2024,6,714
6,2024,7,710
7,2024,8,707
8,2024,9,684
9,2024,10,709


In [ ]:
iceberg_sql(f"""select  circuit_id, t_stamp
            from ts 
            where year=2024 and month=1 and is_pv=True 
            group by circuit_id, t_stamp
            having count(t_stamp) > 1
            limit 10
            """)


,circuit_id,t_stamp


In [50]:
iceberg_sql(f"""select *
            from ts 
            where year=2024 and month=10 and is_pv=True 
            and circuit_id = 373631 and t_stamp = TIMESTAMP '2024-10-16 01:20:00'
            """)


,circuit_id,t_stamp,power,energy,energy_reactive,energy_import,energy_export,energy_reactive_import,energy_reactive_export,power_factor,voltage,current,year,month,is_pv,postcode
0,373631,2024-10-16 01:20:00,6904.5267,575.3772,13.1731,575.3772,0.0,13.1731,0.0,0.999476,242.4,30.338,2024,10,True,5244


In [45]:
hive_sql(f"""select *
            from ts 
            where year=2024 and month=10 and is_pv=True 
            and circuit_id = 373631 and t_stamp = TIMESTAMP '2024-10-16 01:20:00'
            """)


,circuit_id,t_stamp,power,energy,energy_reactive,energy_import,energy_export,energy_reactive_import,energy_reactive_export,power_factor,voltage,current,year,month,is_pv
0,373631,2024-10-16 01:20:00,6904.5267,575.3772,13.1731,575.3772,0.0,13.1731,0.0,0.999476,242.4,30.338,2024,10,True


In [94]:
hive_sql(f"""select count( t_stamp) from ts where year=2024 and month=12 and is_pv=True limit 10""")


,_col0
0,352660646


In [75]:
iceberg_sql(f"""with data as (
                        select site_id, t_stamp, sum(power*circuit_polarity/1000) as P_kW, 
                            sum(energy_reactive*circuit_polarity/1000*12) as Q_kvar, 
                                avg(voltage) as V, ac_capacity_kw
                    from 
                    (select circuit_id, t_stamp, power, energy_reactive, voltage 
                    from ts where year=2024 and month=11 and is_pv=True and voltage > 0 and voltage < 300) as ts1
                    inner join meta_up23c as m on ts1.circuit_id = m.circuit_id
                    group by site_id, t_stamp, ac_capacity_kw)
                    select * from data limit 1000
                    """)

,site_id,t_stamp,P_kW,Q_kvar,V,ac_capacity_kw
0,1555410224,2024-11-30 23:35:00,2.895290,-0.442740,242.20,10.0
1,1555410224,2024-11-30 17:35:00,-0.004147,0.059066,241.75,10.0
2,1555410224,2024-11-30 15:15:00,-0.004130,0.060280,244.10,10.0
3,1555410224,2024-11-30 12:55:00,-0.004173,0.059630,242.85,10.0
4,1555410224,2024-11-30 05:15:00,5.620537,-1.619330,245.75,10.0
...,...,...,...,...,...,...
995,1555410224,2024-11-11 06:10:00,3.182053,-0.759553,243.25,10.0
996,1555410224,2024-11-11 03:10:00,5.209433,-1.774690,246.35,10.0
997,1555410224,2024-11-11 01:50:00,5.021500,-1.868400,245.60,10.0
998,1555410224,2024-11-11 00:30:00,5.280233,-1.912177,246.80,10.0


In [ ]:
iceberg_sql(f"""with data as (
                        select site_id, t_stamp, sum(power*circuit_polarity/1000) as P_kW, 
                            sum(energy_reactive*circuit_polarity/1000*12) as Q_kvar, 
                                avg(voltage) as V, ac_capacity_kw
                    from ts as ts1
                    inner join meta_up23c as m on ts1.circuit_id = m.circuit_id
                        where year=2024 and month=11 and ts1.is_pv=True and voltage > 0 and voltage < 300
                    group by site_id, t_stamp, ac_capacity_kw
                   )
                    select * from data limit 10
                    """)

,site_id,t_stamp,P_kW,Q_kvar,V,ac_capacity_kw
0,1555410224,2024-11-30 21:35:00,1.768553,-0.424880,241.65,10.0
1,1555410224,2024-11-30 20:35:00,0.784443,-0.189520,241.05,10.0
2,1555410224,2024-11-30 17:15:00,-0.004183,0.060497,244.55,10.0
3,1555410224,2024-11-30 12:15:00,-0.004193,0.059810,242.95,10.0
4,1555410224,2024-11-30 07:35:00,1.148700,0.082717,240.45,10.0
...,...,...,...,...,...,...
995,1555410224,2024-11-11 16:30:00,-0.004370,0.059200,241.70,10.0
996,1555410224,2024-11-11 14:50:00,-0.004410,0.060097,243.20,10.0
997,1555410224,2024-11-11 13:10:00,-0.004437,0.059113,241.65,10.0
998,1555410224,2024-11-11 05:10:00,6.672080,-1.785070,246.20,10.0


In [65]:
iceberg_sql("select * from meta_up23c limit 2")

,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,...,circuit_type,is_pv,min_time,max_time,v_95,v_05,m_id,n_long,n_lat,distance_km
0,1944472430,NSW,2502,150.85,-34.47,Endeavour,5.18,5.0,3.0,2021-08-09,...,pv_site_net,True,2024-07-23 04:45:00,2024-07-24 07:05:00,244.3,238.65,M43,150.84,-34.46,1.569777
1,1944472430,NSW,2502,150.85,-34.47,Endeavour,5.18,5.0,3.0,2021-08-09,...,ac_load_net,False,2024-07-23 04:45:00,2024-07-24 07:05:00,244.3,238.65,M43,150.84,-34.46,1.569777


In [18]:
iceberg_sql("""
            with data as (
                select ts.circuit_id, t_stamp, power as P_kW, 
                       energy_reactive*circuit_polarity/1000*12 as Q_kvar, voltage as V
            from ts left join meta_up23c as m on ts.circuit_id = m.circuit_id
            where year=2024 and month=1 and ts.is_pv=True and ts.circuit_id = 467634)
            select * from data order by t_stamp limit 10
            """)

,circuit_id,t_stamp,P_kW,Q_kvar,V
0,467634,2024-01-01 00:00:00,4473.9233,None,241.45
1,467634,2024-01-01 00:05:00,4523.3200,None,241.25
2,467634,2024-01-01 00:10:00,4535.8167,None,241.65
3,467634,2024-01-01 00:15:00,4589.9300,None,242.35
4,467634,2024-01-01 00:20:00,4619.6267,None,241.65
5,467634,2024-01-01 00:25:00,4716.0133,None,241.90
6,467634,2024-01-01 00:30:00,4767.7167,None,242.15
7,467634,2024-01-01 00:35:00,4758.5533,None,242.00
8,467634,2024-01-01 00:40:00,4803.2267,None,242.10
9,467634,2024-01-01 00:45:00,4833.1000,None,243.00
